In [1]:
!pip install streamlit pandas numpy scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 92.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 117.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [streamlit]


In [9]:
import os
import sys
import time
import shutil
import subprocess
import urllib.request
import pandas as pd
import numpy as np
import re
import ast

# 1. COMPLETE STREAMLIT APPLICATION

app_code = r'''
import streamlit as st
import pandas as pd
import numpy as np
import re
import ast
import html

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

st.set_page_config(
    page_title="TED Talks Discovery Engine",
    layout="wide"
)

# PREMIUM BEAUTIFUL UI STYLING (NO EMOJIS, TIGHT SPACING)
st.markdown("""
<style>
/* Global Overrides */
[data-testid="stHeader"],
[data-testid="stAppHeader"] {
    background: transparent !important;
    height: 3.5rem !important;
    min-height: 3.5rem !important;
    display: flex !important;
    visibility: visible !important;
    opacity: 1 !important;
    position: relative !important;
    overflow: visible !important;
    z-index: 999999 !important;
}

/* Hide Streamlit Sidebar elements permanently */
[data-testid="stSidebar"],
section[data-testid="stSidebar"],
[data-testid="collapsedControl"],
[data-testid="stSidebarCollapsedControl"],
[data-testid="stSidebarCollapseButton"],
button[aria-label="Open sidebar"],
button[title="Open sidebar"] {
    display: none !important;
    visibility: hidden !important;
    width: 0px !important;
}

.stApp {
    background-color: #f7fafc;
    color: #2d3748;
    font-family: 'Inter', -apple-system, sans-serif;
}
.block-container {
    max-width: 1200px;
    padding-top: 0.5rem !important;
    padding-bottom: 1.5rem !important;
}

/* Gradient Hero Banner */
.hero-container {
    background: linear-gradient(135deg, #14213d 0%, #1a365d 50%, #168397 100%);
    border-radius: 12px;
    padding: 20px 30px;
    margin-bottom: 20px;
    color: white;
    box-shadow: 0 4px 15px rgba(20, 33, 61, 0.1);
}
.hero-title {
    font-size: 28px;
    font-weight: 800;
    margin: 0;
    color: #ffffff;
    letter-spacing: -0.5px;
}
.hero-subtitle {
    font-size: 14px;
    color: #cbd5e0;
    margin-top: 4px;
    margin-bottom: 0;
    font-weight: 400;
}

/* Cards & Sections with Reduced Spacing */
.card-container {
    background: white;
    border-radius: 8px;
    border: 1px solid #edf2f7;
    padding: 16px 20px;
    box-shadow: 0 2px 4px rgba(0,0,0,0.02);
    transition: transform 0.2s, box-shadow 0.2s;
    margin-bottom: 12px;
}
.card-container:hover {
    transform: translateY(-1px);
    box-shadow: 0 6px 10px rgba(0,0,0,0.05);
}
.main-recommendation-card {
    border-left: 5px solid #168397;
}

/* Headers */
.section-title {
    color: #14213d;
    font-size: 18px;
    font-weight: 750;
    margin-top: 15px;
    margin-bottom: 12px;
    display: flex;
    align-items: center;
    gap: 8px;
    border-bottom: 2px solid #e2e8f0;
    padding-bottom: 4px;
}

/* Typography inside cards */
.talk-title-text {
    color: #14213d;
    font-size: 17px;
    font-weight: 700;
    line-height: 1.3;
    margin-bottom: 4px;
}
.speaker-text {
    color: #168397;
    font-size: 14px;
    font-weight: 600;
    margin-bottom: 6px;
}
.desc-text {
    color: #4a5568;
    font-size: 13.5px;
    line-height: 1.5;
    margin-bottom: 10px;
}

/* Badges & Metrics */
.pill-tag {
    display: inline-block;
    background: #edf2f7;
    color: #4a5568;
    padding: 3px 8px;
    margin-right: 4px;
    margin-top: 4px;
    border-radius: 12px;
    font-size: 11px;
    font-weight: 500;
}
.pill-tag-topic {
    background: #e6fffa;
    color: #00685b;
}
.similarity-badge {
    background: #ebf8ff;
    color: #2b6cb0;
    font-weight: 700;
    padding: 4px 8px;
    border-radius: 6px;
    font-size: 12px;
    display: inline-flex;
    align-items: center;
    gap: 4px;
}
</style>
""", unsafe_allow_html=True)

# HEADER
st.markdown("""
<div class="hero-container">
    <div class="hero-title">TED Talks Discovery Engine</div>
    <div class="hero-subtitle">Harnessing text similarities and semantics to locate ideas worth spreading.</div>
</div>
""", unsafe_allow_html=True)

# LOAD DATASET
@st.cache_data
def load_data():
    return pd.read_csv("ted_talks_final_cleaned.csv")

try:
    df = load_data()
except Exception as e:
    st.error(f"Dataset loading error: {e}")
    st.stop()

if "title" not in df.columns:
    st.error("The dataset must contain a title column.")
    st.stop()

df = df.copy()
df["title"] = df["title"].fillna("").astype(str)
df = df[df["title"].str.strip() != ""].reset_index(drop=True)

if df.empty:
    st.error("No valid TED Talks found.")
    st.stop()

# TEXT CONVERSION
def convert_to_text(value):
    if value is None:
        return ""
    if isinstance(value, (list, tuple)):
        return " ".join(map(str, value))
    if isinstance(value, dict):
        return " ".join(map(str, value.values()))
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    value = str(value)
    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, (list, tuple)):
            return " ".join(map(str, parsed))
        if isinstance(parsed, dict):
            return " ".join(map(str, parsed.values()))
    except (ValueError, SyntaxError):
        pass
    return value

# TEXT PREPROCESSING
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

text_columns = ["title", "topics", "description", "transcript"]
for column in text_columns:
    if column not in df.columns:
        df[column] = ""
    df[column] = df[column].apply(convert_to_text)

df["combined_text"] = (
    df["title"] + " " +
    df["topics"] + " " +
    df["description"] + " " +
    df["transcript"]
)
df["combined_text"] = df["combined_text"].apply(clean_text)

# TF-IDF MODEL
@st.cache_resource
def create_tfidf(text_data):
    vectorizer = TfidfVectorizer(stop_words="english", max_features=10000)
    matrix = vectorizer.fit_transform(text_data)
    return vectorizer, matrix

try:
    vectorizer, tfidf_matrix = create_tfidf(df["combined_text"].tolist())
except ValueError:
    st.error("Unable to build the TF-IDF model.")
    st.stop()

# HELPER FUNCTIONS
def display_value(row, column):
    if column not in row.index:
        return "Not available"
    value = row[column]
    if pd.isna(value) or str(value).strip() == "":
        return "Not available"
    return str(value)

def get_topic_list(value):
    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, (list, tuple)):
            return [str(x).strip() for x in parsed]
    except (ValueError, SyntaxError):
        pass
    return [x.strip() for x in str(value).split(",") if x.strip()]

# STATISTICS
total_talks = len(df)
total_speakers = df["speaker_1"].nunique() if "speaker_1" in df.columns else 0
all_topics = []
for value in df["topics"]:
    all_topics.extend(get_topic_list(value))
total_topics = len(set(all_topics))

# OVERVIEW CARDS
st.markdown("<div class='section-title'>Library Analytics</div>", unsafe_allow_html=True)
c1, c2, c3 = st.columns(3)
with c1:
    st.markdown(f"""
    <div class="card-container" style="text-align:center;">
        <div style="color:#718096; font-size:11px; font-weight:600; text-transform:uppercase;">Total Indexed Talks</div>
        <div style="color:#14213d; font-size:24px; font-weight:800; margin-top:2px;">{total_talks:,}</div>
    </div>
    """, unsafe_allow_html=True)
with c2:
    st.markdown(f"""
    <div class="card-container" style="text-align:center;">
        <div style="color:#718096; font-size:11px; font-weight:600; text-transform:uppercase;">Unique Speakers</div>
        <div style="color:#14213d; font-size:24px; font-weight:800; margin-top:2px;">{total_speakers:,}</div>
    </div>
    """, unsafe_allow_html=True)
with c3:
    st.markdown(f"""
    <div class="card-container" style="text-align:center;">
        <div style="color:#718096; font-size:11px; font-weight:600; text-transform:uppercase;">Covered Topics</div>
        <div style="color:#14213d; font-size:24px; font-weight:800; margin-top:2px;">{total_topics:,}</div>
    </div>
    """, unsafe_allow_html=True)

# SEARCH REFERENCE TALK
st.markdown("<div class='section-title'>Choose a Reference Talk</div>", unsafe_allow_html=True)
available_indices = df.index.tolist()
selected_index = st.selectbox(
    "Select a seed TED Talk to find recommendations for:",
    options=available_indices,
    format_func=lambda i: f" {df.loc[i, 'speaker_1']} - {df.loc[i, 'title']}"
)

# IN-PAGE SETTINGS DIRECTLY UNDER DROP BOX
s_col1, s_col2 = st.columns(2)
with s_col1:
    number = st.selectbox("Number of Recommendations", [5, 10, 15], index=0)
with s_col2:
    sort_option = st.selectbox("Re-rank Results By", ["Similarity", "Views", "Duration"], index=0)

topic_filter = "All Topics"

# SELECTED TALK DISPLAY
selected_talk = df.loc[selected_index]
talk_topics = get_topic_list(selected_talk["topics"])
tags_html = "".join(f'<span class="pill-tag pill-tag-topic">{html.escape(topic)}</span>' for topic in talk_topics[:8])

st.markdown(f"""
<div class="card-container main-recommendation-card">
    <div class="talk-title-text">{html.escape(selected_talk['title'])}</div>
    <div class="speaker-text">Speaker: {html.escape(display_value(selected_talk, 'speaker_1'))}</div>
    <div class="desc-text">{html.escape(display_value(selected_talk, 'description'))}</div>
    <div style="margin-top:10px;">{tags_html}</div>
</div>
""", unsafe_allow_html=True)

talk_url = display_value(selected_talk, "url")
if talk_url.startswith(("https://", "http://")):
    st.link_button("Watch Selected Talk on TED.com", talk_url)

# RECOMMENDATIONS LOGIC
def get_recommendations(selected_index, number, topic_filter, sort_option):
    selected_vector = tfidf_matrix[selected_index]
    scores = cosine_similarity(selected_vector, tfidf_matrix).flatten()
    results = df.copy()
    results["similarity_score"] = scores
    results = results.drop(index=selected_index)
    results = results.sort_values("similarity_score", ascending=False)
    return results.head(number)

st.markdown("<div class='section-title'>Recommended Ideas for You</div>", unsafe_allow_html=True)
recommendations = get_recommendations(selected_index, number, topic_filter, sort_option)

for rank, (idx, talk) in enumerate(recommendations.iterrows(), start=1):
    title = html.escape(str(talk["title"]))
    speaker = html.escape(display_value(talk, "speaker_1"))
    description = html.escape(display_value(talk, "description"))
    score = float(talk["similarity_score"])

    st.markdown(f"""
    <div class="card-container">
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:12px;">
            <span style="color:#168397; font-size:12px; font-weight:800; letter-spacing:1px;">RECOMMENDATION {rank}</span>
            <span class="similarity-badge">Match Score: {score:.2%}</span>
        </div>
        <div class="talk-title-text">{title}</div>
        <div class="speaker-text">Speaker: {speaker}</div>
        <div class="desc-text">{description}</div>
    </div>
    """, unsafe_allow_html=True)

    recommendation_url = display_value(talk, "url")
    if recommendation_url.startswith(("https://", "http://")):
        st.link_button(f"Watch Recommendation {rank} on TED", recommendation_url, key=f"watch_{idx}_{rank}")
'''

# 2. SAVE SCRIPT
with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("✅ app.py created successfully!")

# 3. VERIFY DATASET
if not os.path.exists("ted_talks_final_cleaned.csv"):
    raise FileNotFoundError("ted_talks_final_cleaned.csv not found.")

# 4. START SERVER
subprocess.run("fuser -k 8501/tcp", shell=True, capture_output=True)

log_file = open("/content/logs.txt", "w")

# Dynamically resolve absolute path to active virtual env streamlit binary
streamlit_binary = shutil.which("streamlit")
if not streamlit_binary:
    streamlit_binary = "streamlit"

process = subprocess.Popen([
    streamlit_binary, "run", "app.py",
    "--server.port", "8501", "--server.headless", "true"
], stdout=log_file, stderr=subprocess.STDOUT)

print("⌓ Starting Streamlit...")
server_ready = False
for attempt in range(30):
    if process.poll() is not None:
        break
    try:
        urllib.request.urlopen("http://127.0.0.1:8501/_stcore/health", timeout=2)
        server_ready = True
        break
    except:
        time.sleep(2)

if not server_ready:
    # Print logs to make diagnosis easier
    if os.path.exists("/content/logs.txt"):
        with open("/content/logs.txt", "r") as lf:
            print("--- Streamlit Startup Logs ---")
            print(lf.read())
    raise RuntimeError("Streamlit failed to start. Check /content/logs.txt")

print("✅ Streamlit server started successfully!")

✅ app.py created successfully!
⌓ Starting Streamlit...


✅ Streamlit server started successfully!


In [10]:
import os
import sys
import subprocess

# Automatically check and install pyngrok if not available
try:
    from pyngrok import ngrok
except ModuleNotFoundError:
    print("Installing missing package 'pyngrok'...")
    subprocess.run([sys.executable, "-m", "pip", "install", "pyngrok"], check=True)
    from pyngrok import ngrok

# Configure ngrok auth token
AUTHTOKEN = "3LYAnuqSN8ev3OikqEHqukZ84Zc_TTuUzXmZGRph4UYsusu2"
try:
    ngrok.set_auth_token(AUTHTOKEN)
except Exception as e:
    print(f"Token assignment notice: {e}")

# Kill existing tunnels if any
tunnels = ngrok.get_tunnels()
for t in tunnels:
    ngrok.disconnect(t.public_url)

# Open port 8501 for Streamlit
tunnel = ngrok.connect("8501")
print("====================================")
print("🚀 YOUR APPLICATION IS ONLINE!")
print("====================================")
print(f"Open Link: {tunnel.public_url}")
print("====================================")

Token assignment notice: An error occurred when saving the auth token: b'Authtoken already set in configuration file: /root/.config/ngrok/ngrok.yml\n'
🚀 YOUR APPLICATION IS ONLINE!
Open Link: https://thorn-capable-fernlike.ngrok-free.dev
